In [ ]:
import pandas as pd

# ==========================================
# 0. TẢI DỮ LIỆU
# ==========================================
print("Đang tải dữ liệu...")
orders = pd.read_csv('dataset/orders.csv', parse_dates=['order_date'])
products = pd.read_csv('dataset/products.csv')
returns = pd.read_csv('dataset/returns.csv')
order_items = pd.read_csv('dataset/order_items.csv')
customers = pd.read_csv('dataset/customers.csv')
payments = pd.read_csv('dataset/payments.csv')
web_traffic = pd.read_csv('dataset/web_traffic.csv')
geography = pd.read_csv('dataset/geography.csv')

print("-" * 50)

# ==========================================
# 1. TRẢ LỜI 10 CÂU HỎI TRẮC NGHIỆM
# ==========================================

# Q1. Trung vị số ngày giữa hai lần mua liên tiếp (inter-order gap)
orders_sorted = orders.sort_values(by=['customer_id', 'order_date'])
orders_sorted['prev_order_date'] = orders_sorted.groupby('customer_id')['order_date'].shift(1)
orders_sorted['gap'] = (orders_sorted['order_date'] - orders_sorted['prev_order_date']).dt.days
q1_ans = orders_sorted['gap'].dropna().median()
print(f"Q1: Trung vị khoảng cách mua hàng: {q1_ans} ngày")

# Q2. Phân khúc (segment) có tỷ suất lợi nhuận gộp trung bình cao nhất
products['margin'] = (products['price'] - products['cogs']) / products['price']
q2_ans = products.groupby('segment')['margin'].mean().idxmax()
print(f"Q2: Phân khúc có biên lợi nhuận trung bình cao nhất: {q2_ans}")

# Q3. Lý do trả hàng phổ biến nhất cho danh mục Streetwear
ret_prod = pd.merge(returns, products, on='product_id')
streetwear_rets = ret_prod[ret_prod['category'] == 'Streetwear']
q3_ans = streetwear_rets['return_reason'].value_counts().idxmax()
print(f"Q3: Lý do trả hàng phổ biến nhất của Streetwear: {q3_ans}")

# Q4. Nguồn truy cập có tỷ lệ thoát (bounce_rate) trung bình thấp nhất
q4_ans = web_traffic.groupby('traffic_source')['bounce_rate'].mean().idxmin()
print(f"Q4: Nguồn truy cập có tỷ lệ thoát trung bình thấp nhất: {q4_ans}")

# Q5. Tỷ lệ phần trăm order_items có áp dụng khuyến mãi (promo_id không null)
promo_ratio = order_items['promo_id'].notna().mean() * 100
print(f"Q5: Tỷ lệ dòng order_items có áp dụng khuyến mãi: {promo_ratio:.2f}%")

# Q6. Nhóm tuổi có số đơn hàng trung bình / khách hàng cao nhất
# Lọc bỏ các khách hàng có age_group null
valid_customers = customers.dropna(subset=['age_group'])
cust_orders = pd.merge(valid_customers, orders, on='customer_id')
# Tính tổng đơn hàng từng nhóm tuổi
orders_per_age = cust_orders.groupby('age_group')['order_id'].nunique()
# Tính tổng khách hàng từng nhóm tuổi
cust_per_age = valid_customers.groupby('age_group')['customer_id'].nunique()
q6_ans = (orders_per_age / cust_per_age).idxmax()
print(f"Q6: Nhóm tuổi có số đơn trung bình cao nhất: {q6_ans}")

# Q7. Vùng (region) tạo ra tổng doanh thu cao nhất
# Doanh thu từng item = quantity * unit_price - discount_amount
order_items['revenue'] = order_items['quantity'] * order_items['unit_price'] - order_items['discount_amount'].fillna(0)
orders_geo = pd.merge(orders, geography, on='zip', how='inner')
item_orders_geo = pd.merge(order_items, orders_geo, on='order_id', how='inner')
q7_ans = item_orders_geo.groupby('region')['revenue'].sum().idxmax()
print(f"Q7: Vùng tạo doanh thu cao nhất: {q7_ans}")

# Q8. Phương thức thanh toán dùng nhiều nhất cho đơn hàng bị 'cancelled'
cancelled = orders[orders['order_status'] == 'cancelled']
q8_ans = cancelled['payment_method'].value_counts().idxmax()
print(f"Q8: Phương thức thanh toán cho đơn huỷ nhiều nhất: {q8_ans}")

# Q9. Kích cỡ sản phẩm có tỷ lệ trả hàng cao nhất
items_prod = pd.merge(order_items, products, on='product_id')
ret_prod_all = pd.merge(returns, products, on='product_id')

return_rates = {}
for size in ['S', 'M', 'L', 'XL']:
    num_ret = len(ret_prod_all[ret_prod_all['size'] == size])
    num_items = len(items_prod[items_prod['size'] == size])
    # Tỷ lệ = số bản ghi trong returns chia cho số dòng trong order_items
    return_rates[size] = num_ret / num_items if num_items > 0 else 0

q9_ans = max(return_rates, key=return_rates.get)
print(f"Q9: Kích cỡ có tỷ lệ trả hàng cao nhất: {q9_ans} (với tỷ lệ {return_rates[q9_ans]:.4%})")

# Q10. Kế hoạch trả góp (installments) có giá trị thanh toán trung bình cao nhất
q10_ans = payments.groupby('installments')['payment_value'].mean().idxmax()
print(f"Q10: Số kỳ trả góp có giá trị thanh toán trung bình cao nhất: {q10_ans} kỳ")